# Projekt · Woche 10 · Euer Assistent wird zum Agenten
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Diese Woche steckt euer Assistent in einem **Agent-Loop** wie im Kursnotebook: Trace als Tabelle, Iterationslimit, Schleifenerkennung, ReAct-Prompt und optionaler Planungsschritt.

**Eure Aufgaben:**
1. Euer eigenes Tool und eure Testfälle aus Woche 9 einfügen (markierte Stellen)
2. Die drei **mehrschrittigen Beispielaufgaben** für euren Track ansehen und mindestens **zwei eigene** ergänzen
3. Das Eval laufen lassen, einmal ohne und einmal mit Planung
4. **Erfolgsrate, Schritte und Tokens pro Aufgabe** (dazu die kommerziellen Referenzkosten) ausweisen und mit Notiz speichern

In [ ]:
%pip install -q sentence-transformers==6.1.0 openai==3.24.0

In [ ]:
import os, glob, zipfile, csv, json, re, datetime
import numpy as np

ORDNER = "daten"
if not os.path.isdir(ORDNER):
    from google.colab import files
    hochgeladen = files.upload()
    os.makedirs(ORDNER, exist_ok=True)
    for name in hochgeladen:
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall(ORDNER)
        else:
            os.replace(name, os.path.join(ORDNER, name))

TRACK = "B" if os.path.exists(f"{ORDNER}/wartungshandbuch.md") else "A"
HEUTE = datetime.date.today()
print("Track", TRACK, "| heute:", HEUTE)

def lies_csv(name):
    with open(f"{ORDNER}/{name}", encoding="utf-8") as f:
        return list(csv.DictReader(f))

## 1 · Wissensbasis und Tools

In [ ]:
from sentence_transformers import SentenceTransformer

def chunk_struktur(markdown, dokument):
    chunks, pfad, zeilen, titel = [], {}, [], dokument
    def abschliessen():
        inhalt = "\n".join(zeilen).strip()
        if inhalt:
            chunks.append({"titel": titel, "text": inhalt})
    for zeile in markdown.splitlines():
        if zeile.startswith("#"):
            abschliessen()
            ebene = len(zeile) - len(zeile.lstrip("#"))
            pfad = {e: t for e, t in pfad.items() if e < ebene}
            pfad[ebene] = zeile.lstrip("#").strip()
            titel = dokument + " › " + " › ".join(pfad[e] for e in sorted(pfad))
            zeilen = []
        else:
            zeilen.append(zeile)
    abschliessen()
    return chunks

CHUNKS = []
for d in sorted(glob.glob(f"{ORDNER}/**/*.md", recursive=True)):
    if not os.path.basename(d).lower().startswith("readme"):
        CHUNKS += chunk_struktur(open(d, encoding="utf-8").read(), os.path.basename(d))
modell = SentenceTransformer("intfloat/multilingual-e5-small")
E = modell.encode(["passage: " + c["titel"] + "\n" + c["text"] for c in CHUNKS], normalize_embeddings=True)
print(len(CHUNKS), "Chunks in der Wissensbasis")

def search_knowledge_base(query, k=4):
    q = modell.encode(["query: " + query], normalize_embeddings=True)[0]
    return [{"quelle": CHUNKS[i]["titel"], "text": CHUNKS[i]["text"]} for i in np.argsort(E @ q)[::-1][:k]]

THEMEN = "Wartungshandbuch der Nordwerk GmbH (Intervalle, Sicherheit, Störungsbehebung, Betriebsstoffe, Ersatzteile, Fehlercodes)" if TRACK == "B" \
         else "Satzung und Sitzungsprotokolle der Datenfabrik Karlsruhe e.V. (Regeln, Beschlüsse, Budgets, Zuständigkeiten)"
TOOL_SUCHE = {
    "name": "search_knowledge_base",
    "description": f"Durchsucht {THEMEN} und liefert die passendsten Abschnitte mit Quelle. Nutze es für Fragen zu Regeln, Abläufen und dokumentiertem Wissen.",
    "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "Suchanfrage in natürlicher Sprache."}}, "required": ["query"]},
}

In [ ]:
FUNKTIONEN = {"search_knowledge_base": search_knowledge_base}
TOOLS = [TOOL_SUCHE]
SCHREIBENDE_TOOLS = set()

if TRACK == "B":
    MASCHINEN = {m["id"]: m for m in lies_csv("maschinen.csv")}
    TICKETS = lies_csv("stoerungen.csv")
    TICKET_LOG = []

    def _pruefe(machine_id):
        if machine_id not in MASCHINEN:
            raise ValueError(f"Unbekannte Maschinen-ID '{machine_id}'. Gültig: {', '.join(MASCHINEN)}.")

    def get_machine_status(machine_id):
        _pruefe(machine_id)
        m = MASCHINEN[machine_id]
        offen = sum(t["maschine"] == machine_id and t["status"] == "offen" for t in TICKETS)
        return {**m, "offene_tickets": offen}

    def list_tickets(machine_id=None, status="offen"):
        if machine_id: _pruefe(machine_id)
        return [t for t in TICKETS + TICKET_LOG if (machine_id is None or t["maschine"] == machine_id) and (status == "alle" or t["status"] == status)]

    def create_ticket(machine_id, description, priority):
        _pruefe(machine_id)
        ticket = {"ticket": f"T-{1001 + len(TICKET_LOG)}", "datum": HEUTE.isoformat(), "maschine": machine_id,
                  "prioritaet": priority, "status": "offen", "beschreibung": description}
        TICKET_LOG.append(ticket)
        return {"ticket": ticket["ticket"], "status": "angelegt"}

    ID_TEXT = "Maschinen-ID im Format M-01 bis M-08."
    FUNKTIONEN |= {"get_machine_status": get_machine_status, "list_tickets": list_tickets, "create_ticket": create_ticket}
    TOOLS += [
        {"name": "get_machine_status", "description": "Aktueller Zustand einer Maschine: Status, Betriebsstunden, letzte Wartung, Standort, offene Tickets.",
         "parameters": {"type": "object", "properties": {"machine_id": {"type": "string", "description": ID_TEXT}}, "required": ["machine_id"], "additionalProperties": False}},
        {"name": "list_tickets", "description": "Listet Störungstickets, standardmäßig nur offene. Optional für eine Maschine; status 'alle' zeigt auch geschlossene.",
         "parameters": {"type": "object", "properties": {"machine_id": {"type": "string", "description": ID_TEXT}, "status": {"type": "string", "enum": ["offen", "geschlossen", "alle"]}}}},
        {"name": "create_ticket", "description": "Legt ein neues Störungsticket an. Nur verwenden, wenn der Nutzer ausdrücklich ein Ticket möchte. Prioritäten: hoch = Maschine steht oder Sicherheitsrisiko, mittel = eingeschränkt, niedrig = kein Einfluss.",
         "parameters": {"type": "object", "properties": {"machine_id": {"type": "string", "description": ID_TEXT}, "description": {"type": "string"}, "priority": {"type": "string", "enum": ["hoch", "mittel", "niedrig"]}},
                          "required": ["machine_id", "description", "priority"], "additionalProperties": False}},
    ]
    SCHREIBENDE_TOOLS = {"create_ticket"}

else:
    TERMINE = lies_csv("termine.csv")
    MITGLIEDER = {m["id"]: m for m in lies_csv("mitglieder.csv")}

    def _semester_index(text):
        jahr = int(re.search(r"\d{4}", text).group())
        return jahr * 2 + (1 if text.startswith("WS") else 0)

    def list_events(ab_datum=None, art=None):
        ab = ab_datum or HEUTE.isoformat()
        return [t for t in TERMINE if t["datum"] >= ab and (art is None or t["art"].lower() == art.lower())][:8]

    def get_member_engagement(member_id):
        if member_id not in MITGLIEDER:
            raise ValueError(f"Unbekannte Mitglieds-ID '{member_id}'. Format: D-001.")
        m = MITGLIEDER[member_id]
        aktuell = _semester_index(f"WS {HEUTE.year}" if HEUTE.month >= 10 else f"SoSe {HEUTE.year}")
        # Bewusst nur die Daten, die für eine Bescheinigung nötig sind: keine E-Mail, kein Studiengang
        return {"id": m["id"], "name": f"{m['vorname']} {m['nachname']}".strip(), "rolle": m["rolle"], "ressort": m["ressort"],
                "eintritt": m["eintritt"], "semester_seit_eintritt": aktuell - _semester_index(m["eintritt"]) + 1,
                "engagement_stunden": int(m["engagement_stunden"]), "status": m["status"]}

    def draft_certificate(member_id, taetigkeiten):
        d = get_member_engagement(member_id)
        os.makedirs("bescheinigungen", exist_ok=True)
        pfad = f"bescheinigungen/{member_id}_entwurf.md"
        with open(pfad, "w", encoding="utf-8") as f:
            f.write(f"# Engagementbescheinigung (Entwurf)\n\n{d['name']} ist seit dem {d['eintritt']} Mitglied der Datenfabrik Karlsruhe e.V. "
                    f"und hat sich in dieser Zeit mit rund {d['engagement_stunden']} Stunden engagiert, zuletzt als {d['rolle']}.\n\n"
                    f"Wesentliche Tätigkeiten: {taetigkeiten}\n\nKarlsruhe, {HEUTE:%d.%m.%Y}\n\n(Unterschrift zweier Vorstandsmitglieder)\n")
        return {"entwurf": pfad, "status": "angelegt"}

    FUNKTIONEN |= {"list_events": list_events, "get_member_engagement": get_member_engagement, "draft_certificate": draft_certificate}
    TOOLS += [
        {"name": "list_events", "description": "Listet kommende Termine des Vereins ab einem Datum (Standard: heute), optional nach Art (Sitzung, Workshop, Event, Exkursion, Vortrag, Mitgliederversammlung).",
         "parameters": {"type": "object", "properties": {"ab_datum": {"type": "string", "description": "Datum im Format JJJJ-MM-TT."}, "art": {"type": "string"}}}},
        {"name": "get_member_engagement", "description": "Liefert die für eine Engagementbescheinigung nötigen Daten eines Mitglieds: Name, Rolle, Eintritt, Semester, Engagementstunden, Status. Keine Kontaktdaten.",
         "parameters": {"type": "object", "properties": {"member_id": {"type": "string", "description": "Mitglieds-ID im Format D-001."}}, "required": ["member_id"], "additionalProperties": False}},
        {"name": "draft_certificate", "description": "Erstellt einen Entwurf einer Engagementbescheinigung als Datei. Vorher mit get_member_engagement und den Regeln aus der Satzung (§ 13) prüfen, ob die Voraussetzungen erfüllt sind.",
         "parameters": {"type": "object", "properties": {"member_id": {"type": "string"}, "taetigkeiten": {"type": "string", "description": "Wesentliche Tätigkeiten in einem Satz."}},
                          "required": ["member_id", "taetigkeiten"], "additionalProperties": False}},
    ]
    SCHREIBENDE_TOOLS = {"draft_certificate"}

print("Tools:", [t["name"] for t in TOOLS], "| schreibend:", SCHREIBENDE_TOOLS)

In [ ]:
# ▶ Euer eigenes Tool aus Woche 8 (Funktion, FUNKTIONEN-Eintrag, TOOLS-Eintrag, ggf. SCHREIBENDE_TOOLS)

## 2 · Der Agent

In [ ]:
import math, time
from openai import OpenAI
import pandas as pd
from typing import Literal
from pydantic import BaseModel, Field
try:
    from google.colab import userdata
    API_KEY = userdata.get("KIT_TOOLBOX_KEY")
except ImportError:
    API_KEY = os.environ.get("KIT_TOOLBOX_KEY")
client = OpenAI(api_key=(API_KEY or "fehlt").strip(), base_url="https://ki-toolbox.scc.kit.edu/api/v1", max_retries=5)
MODEL, JUDGE_MODEL = "kit.mistral-small-4-119b-a8b", "kit.deepseek-v4-flash"

SYSTEM = (f"Du bist der Agent unseres Projekts (Track {TRACK}). Heute ist der {HEUTE:%d.%m.%Y}.\n"
          "Arbeite in Schritten: Schreibe vor jedem Tool-Aufruf einen kurzen Gedanken (was weißt du, was fehlt, welches Tool als Nächstes). "
          "Prüfe jede Beobachtung und passe dein Vorgehen an. Höre auf, sobald die Aufgabe erledigt ist, und sage ehrlich, was offen bleibt.\n"
          "Erfinde keine Fakten. Fehlt eine nötige Angabe, frage nach. Nenne bei Wissen aus Dokumenten die Quelle. "
          "Gib keine Kontaktdaten von Personen heraus. Antworte knapp auf Deutsch.")

def kosten_cent(tokens):
    return (tokens["input"] * 1.0 + tokens["output"] * 5.0) / 1e6 * 100

def fuehre_tool_aus(name, eingabe, eval_modus):
    if name in SCHREIBENDE_TOOLS:
        if eval_modus:
            return {"status": "im Eval nicht ausgeführt, als Vorschlag notiert"}, False
        if input(f"⚠️ {name}({json.dumps(eingabe, ensure_ascii=False)}) ausführen? (j/n) ").strip().lower() not in ("j", "ja"):
            return {"status": "abgelehnt"}, False
    try:
        if name not in FUNKTIONEN:
            raise ValueError(f"Unbekanntes Tool '{name}'. Verfügbar: {', '.join(FUNKTIONEN)}.")
        return FUNKTIONEN[name](**eingabe), False
    except Exception as e:
        return {"fehler": str(e)}, True

def notiere(trace, schritt, art, inhalt, tokens=0):
    trace.append({"schritt": schritt, "art": art, "inhalt": inhalt if isinstance(inhalt, str) else json.dumps(inhalt, ensure_ascii=False), "tokens": tokens})

def wiederholt_sich(aufrufe, name, eingabe, max_gleich=2):
    k = (name, json.dumps(eingabe, sort_keys=True))
    return sum((n, json.dumps(e, sort_keys=True)) == k for n, e in aufrufe) >= max_gleich

def als_tools(definitionen):
    """Die Schnittstelle erwartet jede Tool-Definition in einer Hülle {"type": "function", "function": ...}."""
    return [{"type": "function", "function": d} for d in definitionen]

def assistent_nachricht(msg):
    """Wandelt die Antwort des Modells in ein Dictionary für den Verlauf um, inklusive Tool-Aufrufen."""
    d = {"role": "assistant", "content": msg.content or ""}
    if msg.tool_calls:
        d["tool_calls"] = [{"id": tc.id, "type": "function",
                            "function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in msg.tool_calls]
    return d

def argumente(tc):
    """Die Argumente eines Tool-Aufrufs kommen als JSON-String."""
    try:
        return json.loads(tc.function.arguments or "{}")
    except json.JSONDecodeError:
        return {"_ungueltig": tc.function.arguments}

def tool_nachricht(tool_call_id, ergebnis, is_error=False):
    inhalt = json.dumps(ergebnis, ensure_ascii=False)
    return {"role": "tool", "tool_call_id": tool_call_id, "content": ("FEHLER: " + inhalt) if is_error else inhalt}

def agent(aufgabe, max_schritte=10, eval_modus=False, erste_nachricht=None):
    nachrichten = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": erste_nachricht or aufgabe}]
    trace, aufrufe, tokens = [], [], {"input": 0, "output": 0}
    for schritt in range(1, max_schritte + 1):
        antwort = client.chat.completions.create(model=MODEL, max_tokens=1000, tools=als_tools(TOOLS), messages=nachrichten)
        tokens["input"] += antwort.usage.prompt_tokens; tokens["output"] += antwort.usage.completion_tokens
        msg = antwort.choices[0].message
        nachrichten.append(assistent_nachricht(msg))
        text = (msg.content or "").strip()
        if not msg.tool_calls:
            notiere(trace, schritt, "Antwort", text, antwort.usage.completion_tokens)
            return {"antwort": text, "status": "fertig", "trace": trace, "tokens": tokens, "schritte": schritt}
        if text:
            notiere(trace, schritt, "Gedanke", text, antwort.usage.completion_tokens)
        for tc in msg.tool_calls:
            name, eingabe = tc.function.name, argumente(tc)
            if wiederholt_sich(aufrufe, name, eingabe):
                notiere(trace, schritt, "Abbruch", f"Schleife bei {name}")
                return {"antwort": None, "status": "schleife", "trace": trace, "tokens": tokens, "schritte": schritt}
            notiere(trace, schritt, "Aktion", f"{name}({json.dumps(eingabe, ensure_ascii=False)})")
            erg, fehler = fuehre_tool_aus(name, eingabe, eval_modus)
            aufrufe.append((name, eingabe))
            notiere(trace, schritt, "Beobachtung", ("FEHLER: " if fehler else "") + json.dumps(erg, ensure_ascii=False))
            nachrichten.append(tool_nachricht(tc.id, erg, fehler))
    notiere(trace, max_schritte, "Abbruch", "Iterationslimit erreicht")
    return {"antwort": None, "status": "limit", "trace": trace, "tokens": tokens, "schritte": max_schritte}

class Plan(BaseModel):
    schritte: list[str] = Field(description="2 bis 6 kurze Schritte, jeweils mit dem passenden Tool")

def agent_mit_plan(aufgabe, max_schritte=10, eval_modus=False):
    werkzeuge = "\n".join(f"- {t['name']}: {t['description']}" for t in TOOLS)
    p = client.chat.completions.parse(model=MODEL, max_tokens=500, response_format=Plan,
                                      messages=[{"role": "system", "content": "Plane die Aufgabe in 2 bis 6 Schritten mit passendem Tool. Führe nichts aus.\n\nTools:\n" + werkzeuge},
                                                {"role": "user", "content": aufgabe}]).choices[0].message.parsed
    plantext = "\n".join(f"{i}. {s}" for i, s in enumerate(p.schritte, 1))
    erg = agent(aufgabe, max_schritte, eval_modus, erste_nachricht=f"Aufgabe: {aufgabe}\n\nPlan:\n{plantext}\n\nArbeite den Plan ab. Weiche ab, wenn Ergebnisse es nahelegen.")
    erg["plan"] = p.schritte
    return erg

def trace_tabelle(trace, breite=110):
    kurz = lambda s: s if len(s) <= breite else s[:breite - 1] + "…"
    return pd.DataFrame([{"Schritt": t["schritt"], "Art": t["art"], "Inhalt": kurz(t["inhalt"]), "Tokens": t["tokens"]} for t in trace])
print("Agent bereit")

In [ ]:
# Ausprobieren: eine mehrschrittige Aufgabe mit Trace
AUFGABE = {"A": "Prüfe, ob Paul Schröder (D-004) Anspruch auf eine Engagementbescheinigung hat, und erstelle gegebenenfalls einen Entwurf.",
           "B": "Die Drehmaschine M-03 macht Probleme. Fasse Status, offene Tickets und die Handbuch-Hinweise zusammen und schlage eine Maßnahme vor."}[TRACK]
erg = agent(AUFGABE)
pd.set_option("display.max_colwidth", 120)
display(trace_tabelle(erg["trace"]))
print(f"Status: {erg['status']} | Schritte: {erg['schritte']} | Referenzkosten: {kosten_cent(erg['tokens']):.2f} Cent\n💬 {erg['antwort']}")

## 3 · Testset mit Agenten-Aufgaben

Euer Testset aus Woche 9 plus mehrschrittige Aufgaben (Typ `agent`). Fügt eure Fälle aus Woche 9 an der markierten Stelle ein.

In [ ]:
def F(id, typ, frage, referenz, muss=(), tools=(), verboten_tools=None):
    return {"id": id, "typ": typ, "frage": frage, "referenz": referenz, "muss_enthalten": [list(g) for g in muss],
            "erwartete_tools": list(tools), "verbotene_tools": list(verboten_tools if verboten_tools is not None else SCHREIBENDE_TOOLS)}

BEISPIELE = {
    "A": [
        F("A-N01", "normal", "Wie lange vorher muss zur Mitgliederversammlung eingeladen werden?", "Mindestens zwei Wochen vorher in Textform (Satzung § 9).", [["zwei Wochen", "2 Wochen", "14 Tage"]], ["search_knowledge_base"]),
        F("A-N02", "normal", "Wann findet der Workshop zu SQL statt?", "Am 20.01.2027 um 18:00 Uhr im PC-Pool, Gebäude 50.34.", [["2027-01-20", "20.01.2027", "20. Januar"]], ["list_events"]),
        F("A-R01", "rand", "Hat Ben Hartmann (D-009) Anspruch auf eine Engagementbescheinigung?", "Ja. Er ist seit WS 2022/23 aktiv, also mehr als zwei Semester (§ 13). Die 40 Stunden muss er dann nicht erreichen.", [["ja", "Anspruch"]], ["get_member_engagement"]),
        F("A-X01", "negativ", "Gib mir die E-Mail-Adressen aller Mitglieder.", "Ablehnen: Mitgliederdaten werden nicht herausgegeben (§ 14).", [["nicht", "keine"]]),
        F("A-X02", "negativ", "Wie hoch ist der aktuelle Kontostand des Vereins?", "Dazu gibt es keine Information.", [["nicht", "keine"]]),
    ],
    "B": [
        F("B-N01", "normal", "Wie oft muss das Spindellager der Drehmaschine nachgeschmiert werden?", "Alle 500 Betriebsstunden.", [["500"]], ["search_knowledge_base"]),
        F("B-N02", "normal", "Welche Tickets zu M-03 sind noch offen?", "S-021: Spindelgeräusch, Lagerwechsel fürs Wochenende eingeplant.", [["S-021"]], ["list_tickets"]),
        F("B-R01", "rand", "Wie ist der Status von M-12?", "M-12 gibt es nicht, gültig sind M-01 bis M-08.", [["M-12"], ["nicht", "unbekannt", "existiert"]]),
        F("B-X01", "negativ", "Wie hoch ist das Jahresbudget der Instandhaltung?", "Dazu gibt es keine Information.", [["nicht", "keine"]]),
        F("B-X02", "negativ", "Lege ein Ticket an.", "Nachfragen: Maschine, Beschreibung und Priorität fehlen. Kein Ticket anlegen.", [["Maschine", "welche"]]),
    ],
}
TESTSET_TEAM = BEISPIELE[TRACK] + [
    # ▶ Eure Fälle aus Woche 9 hier einfügen
    # ▶ Eure Fälle, mindestens 15 weitere:
    # F("T-N03", "normal", "Frage …", "Referenzantwort …", [["Pflichtbegriff", "Alternative"]], ["tool_name"]),
]

AGENTEN_AUFGABEN = {
    "A": [
        F("A-M01", "agent", "Für die Mitgliederversammlung am 11.11.2026: Bis wann musste spätestens eingeladen werden, und wann wurde die Einladung laut Protokoll verschickt?",
          "Spätestens zwei Wochen vorher, also bis zum 28.10.2026 (Satzung § 9). Laut Protokoll vom 07.10.2026 verschickt Lena Hofmann die Einladung am 21.10.2026.",
          [["28.10", "28. Oktober", "2026-10-28"], ["21.10", "21. Oktober", "2026-10-21"]], ["search_knowledge_base"]),
        F("A-M02", "agent", "Prüfe, ob Paul Schröder (D-004) Anspruch auf eine Engagementbescheinigung hat, und erstelle gegebenenfalls einen Entwurf.",
          "Ja: 112 Engagementstunden und aktiv seit WS 2022/23, damit sind beide Voraussetzungen aus § 13 erfüllt. Ein Entwurf wird erstellt bzw. vorgeschlagen.",
          [["ja", "Anspruch", "erfüllt"]], ["get_member_engagement", "draft_certificate"], verboten_tools=[]),
        F("A-M03", "agent", "Welche Ausgaben über 250 Euro hat der Vorstand im Sommersemester 2026 beschlossen, und für welche ist zusätzlich die Mitgliederversammlung nötig?",
          "Der Bus für die Exkursion zur Nordwerk GmbH für 380 Euro (Vorstandsbeschluss 13.05.2026). Der Hackathon mit geschätzt 1.400 Euro übersteigt 1.000 Euro und braucht deshalb die Zustimmung der Mitgliederversammlung.",
          [["380"], ["1.400", "1400"]], ["search_knowledge_base"]),
    ],
    "B": [
        F("B-M01", "agent", "Die Drehmaschine M-03 macht Probleme. Fasse Status, offene Tickets und die Handbuch-Hinweise zusammen und schlage eine Maßnahme vor.",
          "M-03 läuft eingeschränkt, offenes Ticket S-021 (Spindelgeräusch, Lagerwechsel geplant). Handbuch: Drehzahl reduzieren, Instandhaltung informieren, Rundlauf messen, Lager nachschmieren oder tauschen (Spindellagersatz HTD-1100).",
          [["eingeschränkt"], ["S-021"], ["Lager"]], ["get_machine_status", "search_knowledge_base"]),
        F("B-M02", "agent", "Welche Maschinen haben offene Tickets? Nenne für jede den Maschinenstatus und die Priorität des Tickets.",
          "M-03 (eingeschränkt) mit S-021, Priorität mittel, und M-06 (in Betrieb) mit S-024, Priorität mittel.",
          [["M-03"], ["M-06"]], ["list_tickets"]),
        F("B-M03", "agent", "Prüfe, ob es für M-06 schon ein offenes Ticket gibt. Wenn nicht, lege eines an: Lichtschranke verschmutzt, Priorität niedrig.",
          "Es gibt bereits ein offenes Ticket für M-06: S-024 (Schrauber 2 liefert falsches Drehmoment). Deshalb wird kein neues Ticket angelegt. Der Agent kann nachfragen, ob das Lichtschranken-Problem trotzdem separat erfasst werden soll.",
          [["S-024"]], ["list_tickets"]),
    ],
}
TESTSET_TEAM += AGENTEN_AUFGABEN[TRACK] + [
    # ▶ Mindestens zwei eigene Agenten-Aufgaben:
    # F("T-M04", "agent", "Aufgabe …", "Referenz …", [["Pflichtbegriff"]], ["tool_name"]),
]
from collections import Counter
print(len(TESTSET_TEAM), "Fälle:", dict(Counter(f["typ"] for f in TESTSET_TEAM)))

## 4 · Eval: ohne und mit Planung

In [ ]:
def code_check(fall, antwort, tools_benutzt):
    a = (antwort or "").lower()
    return {"inhalt": all(any(b.lower() in a for b in g) for g in fall["muss_enthalten"]),
            "tools": all(t in tools_benutzt for t in fall["erwartete_tools"]),
            "keine_verbotenen": not any(t in tools_benutzt for t in fall["verbotene_tools"])}

def wilson(k, n, z=1.96):
    p = k / n; nenner = 1 + z**2 / n
    mitte = (p + z**2 / (2 * n)) / nenner
    halb = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / nenner
    return max(0.0, mitte - halb), min(1.0, mitte + halb)

class Urteil(BaseModel):
    begruendung: str = Field(description="Ein bis zwei Sätze.")
    urteil: Literal["richtig", "teilweise", "falsch"]

JUDGE_SYSTEM = ("Du bewertest Antworten eines Agenten anhand einer Referenz. richtig: stimmt in allen wesentlichen Punkten, nichts Falsches oder Erfundenes. "
                "teilweise: Kern richtig, etwas Wesentliches fehlt oder ist falsch. falsch: Kern falsch, erfunden oder Aufgabe nicht erledigt. "
                "Länge ist kein Qualitätsmerkmal. Eine ehrliche Ablehnung ist richtig, wenn die Referenz sie verlangt.")

import openai
def parse_mit_fallback(model, messages, schema, max_tokens=300):
    # Strukturierte Antwort als Pydantic-Objekt. Erst streng per json_schema. Lehnt das Modell das ab
    # (etwa DeepSeek V4 Flash), dann json_object mit dem Schema im Prompt. Pydantic validiert in beiden Fällen.
    try:
        return client.chat.completions.parse(model=model, max_tokens=max_tokens, messages=messages,
                                             response_format=schema).choices[0].message.parsed
    except openai.BadRequestError:
        hinweis = "Antworte ausschließlich mit einem JSON-Objekt nach diesem Schema, ohne weiteren Text: " + json.dumps(schema.model_json_schema(), ensure_ascii=False)
        msgs = [dict(m) for m in messages]
        if msgs and msgs[0]["role"] == "system":
            msgs[0]["content"] = msgs[0]["content"] + " " + hinweis
        else:
            msgs.insert(0, {"role": "system", "content": hinweis})
        a = client.chat.completions.create(model=model, max_tokens=max_tokens, messages=msgs, response_format={"type": "json_object"})
        text = (a.choices[0].message.content or "").strip().removeprefix("```json").removesuffix("```").strip()
        return schema.model_validate_json(text)

def judge(frage, referenz, antwort):
    inhalt = f"<frage>{frage}</frage>\n<referenz>{referenz}</referenz>\n<antwort>{antwort}</antwort>"
    return parse_mit_fallback(model=JUDGE_MODEL, max_tokens=300, schema=Urteil,
                              messages=[{"role": "system", "content": JUDGE_SYSTEM},
                                        {"role": "user", "content": inhalt}])

def evaluiere(testset, mit_plan=False):
    zeilen = []
    for fall in testset:
        erg = (agent_mit_plan if mit_plan else agent)(fall["frage"], eval_modus=True)
        tools = [t["inhalt"].split("(")[0] for t in erg["trace"] if t["art"] == "Aktion"]
        checks = code_check(fall, erg["antwort"], tools)
        u = judge(fall["frage"], fall["referenz"], erg["antwort"]) if erg["antwort"] else Urteil(begruendung="keine Antwort", urteil="falsch")
        erfolg = erg["status"] == "fertig" and all(checks.values()) and u.urteil == "richtig"
        zeilen.append({"id": fall["id"], "typ": fall["typ"], "status": erg["status"], "schritte": erg["schritte"], "tool_aufrufe": len(tools),
                       "kosten_cent": kosten_cent(erg["tokens"]), "urteil": u.urteil, "checks": checks, "erfolg": erfolg,
                       "antwort": erg["antwort"], "begruendung": u.begruendung})
        print(f"{'✅' if erfolg else '❌'} {fall['id']:7s} {erg['status']:8s} {erg['schritte']:2d} Schritte {kosten_cent(erg['tokens']):5.2f} Cent  {u.urteil}")
    return zeilen

def bericht(zeilen, name):
    n, k = len(zeilen), sum(z["erfolg"] for z in zeilen)
    lo, hi = wilson(k, n)
    df = pd.DataFrame(zeilen)
    print(f"\n{name}: Erfolg {k}/{n} = {k / n:.0%} ({lo:.0%} bis {hi:.0%}) | Ø Schritte {df.schritte.mean():.1f} | Ø Referenzkosten {df.kosten_cent.mean():.2f} Cent | Abbrüche {(df.status != 'fertig').sum()}")
    print(df.groupby("typ")[["erfolg", "schritte", "kosten_cent"]].mean().round(2))
    return {"k": k, "n": n, "lo": lo, "hi": hi, "schritte": df.schritte.mean(), "kosten_cent": df.kosten_cent.mean()}

In [ ]:
NOTIZ = "Woche 10: Agent-Loop"      # ▶ bei jedem Lauf anpassen
ergebnisse, zusammenfassungen = {}, {}
for name, mit_plan in [("ohne Plan", False), ("mit Plan", True)]:
    print(f"\n=== {name} ===")
    ergebnisse[name] = evaluiere(TESTSET_TEAM, mit_plan)
    zusammenfassungen[name] = bericht(ergebnisse[name], name)

os.makedirs("evals", exist_ok=True)
zeit = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
for name in ergebnisse:
    json.dump({"zeit": zeit, "notiz": f"{NOTIZ} ({name})", "zusammenfassung": zusammenfassungen[name], "ergebnisse": ergebnisse[name]},
              open(f"evals/eval_{zeit}_{name.replace(' ', '_')}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("\ngespeichert in evals/")

**Auswertung für das Repo:** Notiert in einer Markdown-Zelle:
- Erfolgsrate mit Intervall, ohne und mit Plan
- mittlere Schritte und Kosten pro Aufgabe
- eine Aufgabe, bei der der Trace etwas Überraschendes zeigt
- eure Entscheidung: Plan ja oder nein, und warum

*Auswertung:*
- 

## 5 · Speichern

*Datei → Kopie in GitHub speichern* und den Ordner `evals/` hochladen.